In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings

In [14]:
warnings.filterwarnings('ignore')

## Função para calcular tabela de frequência de uma variável qualitativa

In [15]:
def tab_freq_quali(data):
    valores = list(data.unique())
    freq = data.value_counts()
    freq_relativa = data.value_counts(normalize=True)*100
    lista_acumulada = []
    for i in range(len(valores)):
        if i == 0:
            lista_acumulada.append(list(freq_relativa)[i])
        else:
            lista_acumulada.append(lista_acumulada[i-1] + list(freq_relativa)[i])
    
    dist_freq = pd.DataFrame({'Frequência':freq, 'Porcentagem (%)':freq_relativa, 'Acumulada (%)':lista_acumulada})
    soma = dist_freq['Frequência'].sum()
    soma_relativa = dist_freq['Porcentagem (%)'].sum()
    dist_freq.loc['Total'] = [soma, soma_relativa, " "]
    return dist_freq

## Função para calcular as faixas de uma variável quantitativa contínua

In [16]:
def faixas(dados, intervalo, estouro_inferior=None, estouro_superior=None):
    i = 0
    j = 0
    classes = []

    if estouro_inferior != None and estouro_superior == None:
        classes.append(-np.inf)
        classes.append(estouro_inferior)
        i = estouro_inferior
        j = dados.max()
        
        while i + intervalo < j:
            s = i + intervalo
            classes.append(s)
            i = s
        classes.append(dados.max() + 0.1)

    if estouro_inferior == None and estouro_superior != None:
        classes.append(dados.min())
        i = dados.min()
        j = estouro_superior

        while i + intervalo < j:
            s = i + intervalo
            classes.append(s)
            i = s
        classes.append(estouro_superior)
        classes.append(np.inf)

    if estouro_inferior == None and estouro_superior == None:
        classes.append(dados.min())
        i = dados.min()
        j = dados.max()
        
        while i + intervalo < j:
            s = i + intervalo
            classes.append(s)
            i = s
        classes.append(dados.max() + 0.1)

    if estouro_inferior != None and estouro_superior != None:
        classes.append(-np.inf)
        classes.append(estouro_inferior)
        i = estouro_inferior
        j = estouro_superior

        while i + intervalo < j:
                s = i + intervalo
                classes.append(s)
                i = s
        classes.append(estouro_superior)
        classes.append(np.inf)
    
    return classes
    

## Função para gerar os labels das faixas de uma tabela de frequência de uma variável contínua

In [17]:
def labels(dados, intervalo, estouro_inferior=None, estouro_superior=None):
    classes = faixas(dados, intervalo, estouro_inferior, estouro_superior)
    labels = []
    if classes[0] == -np.inf and classes[len(classes)-1] != np.inf:
        labels.append('<' + str(estouro_inferior))
        for i in range(len(classes)-3):
            labels.append('[' + str(classes[i+1]) + ', ' + str(classes[i+2]) + ')')
        labels.append('[' + str(classes[len(classes)-2]) + ', ' + str(dados.max()) + ']')
        return labels
    if classes[0] != -np.inf and classes[len(classes)-1] == np.inf:
        for i in range(len(classes)-2):
            labels.append('[' + str(classes[i]) + ', ' + str(classes[i+1]) + ')')
        labels.append('≥' + str(estouro_superior))
        return labels
    if classes[0] == -np.inf and classes[len(classes)-1] == np.inf:
        labels.append('<' + str(estouro_inferior))
        for i in range(len(classes)-3):
            labels.append('[' + str(classes[i+1]) + ', ' + str(classes[i+2]) + ')')
        labels.append('≥' + str(estouro_superior))
        return labels
    if classes[0] != -np.inf and classes[len(classes)-1] != np.inf:
        for i in range(len(classes)-2):
            labels.append('[' + str(classes[i]) + ', ' + str(classes[i+1]) + ')')
        labels.append('[' + str(classes[i+1])  + ', ' + str(dados.max()) + ']')
        return labels


## Função para calcular a tabela de frequênicia de um variável quantitativa contínua

In [18]:
def tab_freq(dados, intervalo, estouro_inferior=None, estouro_superior=None):
    freq = pd.value_counts(pd.cut(x = dados, bins = faixas(dados, intervalo, estouro_inferior, estouro_superior),
                                 labels = labels(dados, intervalo, estouro_inferior, estouro_superior),
                                  include_lowest=True, right=False), sort=False)
    freq_relativa = pd.value_counts(pd.cut(x = dados, bins = faixas(dados, intervalo, estouro_inferior, estouro_superior),
                                           labels = labels(dados, intervalo, estouro_inferior, estouro_superior),
                                           include_lowest=True, right=False), sort=False, normalize=True)*100
    dist_freq = pd.DataFrame({'Frequência': freq, 'Porcentagem (%)': freq_relativa})
    dist_freq['Acumulada (%)'] = dist_freq['Porcentagem (%)'].cumsum()
    soma_freq = dist_freq['Frequência'].sum()
    soma_freq_relativa = dist_freq['Porcentagem (%)'].sum()
    dist_freq.loc['Total'] = [soma_freq, soma_freq_relativa, ' ']
    return  dist_freq


## Função para calcular taxas em de uma variável binária em relação a uma variável qualitativa

In [19]:
def taxas_quali(linhas, colunas, valores):
    tab_taxas = pd.crosstab(index=linhas, columns=colunas, values=valores, aggfunc='count', margins=True)
    #percent = []
    #for i in range(len(tab_taxas)):
        #percent.append(((tab_taxas[1][i]/tab_taxas['All'][i])*100).round(2))
    tab_taxas['Taxa (%)'] = ((tab_taxas[1]/tab_taxas['All'])*100).round(2)
    return tab_taxas